# 09. Stage-0 Model Comparison

This notebook compares four models for the Stage-0 volatility-prediction task:

- Logistic Regression
- Random Forest
- XGBoost
- Feedforward Neural Network

All models are evaluated on the same chronological test set covering 2023–2026. The models, preprocessing objects, and classification thresholds are loaded from the artifacts saved during the preceding experiments.

The test set is used only for final evaluation. No hyperparameters or classification thresholds are selected using test performance.

## 1. Project Setup and Imports.

### 1.1: Define Project Path

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

STAGE0_DATA_FILE = (
    PROCESSED_DATA_DIR / "spy_stage0_features.csv"
)

### 1.2: Import Libraries

In [2]:
import joblib
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

# tensorflow
from tensorflow.keras.models import load_model

# project modules
from src.evaluation import classification_metrics

## 2. Load the Stage-0 Dataset

We use the same processed feature dataset as the preceding model experiments. The target column is separated from the predictor features, and the chronological split is reconstructed using the original date boundaries.

In [3]:
stage0_data = pd.read_csv(
    STAGE0_DATA_FILE,
    index_col=0,
    parse_dates=True,
)

FEATURE_COLUMNS = [
    column for column in stage0_data.columns
    if column != "target"
]

stage0_data.head()

,log_return,abs_return,squared_return,vol_5d,vol_10d,vol_20d,vol_60d,vol_ratio_5_20,vol_ratio_20_60,abs_return_lag_1,...,h0_mean_persistence,h0_max_persistence,h1_total_persistence,h1_mean_persistence,h1_max_persistence,h1_n_features,h1_l1_norm,h1_l2_norm,h1_sup_norm,target
date,,,,,,,,,,,,,,,,,,,,,
2010-03-31,-0.003413,0.003413,0.000012,0.003322,0.004271,0.005152,0.008848,0.644720,0.582289,0.000681,...,0.003322,0.007881,0.000868,0.000868,0.000868,1,1.883799e-07,0.000007,0.000434,0
2010-04-01,0.006815,0.006815,0.000046,0.004449,0.004781,0.005330,0.008885,0.834670,0.599885,0.003413,...,0.002969,0.007881,0.000868,0.000868,0.000868,1,1.883799e-07,0.000007,0.000434,0
2010-04-05,0.008116,0.008116,0.000066,0.005735,0.005184,0.004651,0.008946,1.233187,0.519877,0.006815,...,0.002615,0.004491,0.000868,0.000868,0.000868,1,1.883799e-07,0.000007,0.000434,0
2010-04-06,0.002355,0.002355,0.000006,0.005098,0.004958,0.004680,0.008934,1.089330,0.523846,0.008116,...,0.002702,0.004491,0.000688,0.000344,0.000555,2,8.148190e-08,0.000004,0.000278,0
2010-04-07,-0.005729,0.005729,0.000033,0.005698,0.004791,0.004838,0.008955,1.177664,0.540292,0.002355,...,0.002828,0.005156,0.001775,0.000444,0.001358,4,4.757887e-07,0.000014,0.000679,1


### 3. Chronological Split

In [4]:
TRAIN_END = "2020-12-31"
VALIDATION_END = "2022-12-31"

test_mask = stage0_data.index > VALIDATION_END
test_data = stage0_data.loc[test_mask].copy()

X_test = test_data[FEATURE_COLUMNS]
y_test = test_data["target"]

print("Test period:", X_test.index.min(), "to", X_test.index.max())
print("Test features:", X_test.shape)
print("Test target:", y_test.shape)
print("Positive-class proportion:", y_test.mean())

Test period: 2023-01-03 00:00:00 to 2026-09-01 00:00:00
Test features: (919, 27)
Test target: (919,)
Positive-class proportion: 0.20674646354733406


## 4. Load Saved Models

In [6]:
MODELS_DIR = PROJECT_ROOT / "models"

# Logistic Regression
logistic_model = joblib.load(
    MODELS_DIR / "stage0_logistic_regression.joblib"
)

logistic_scaler = joblib.load(
    MODELS_DIR / "stage0_logistic_regression_scaler.joblib"
)

# Random Forest
random_forest_model = joblib.load(
    MODELS_DIR / "stage0_random_forest.joblib"
)

# XGBoost
xgboost_model = joblib.load(
    MODELS_DIR / "stage0_xgboost.joblib"
)

# Neural Network
neural_network_model = load_model(
    MODELS_DIR / "stage0_neural_network.keras"
)

neural_network_scaler = joblib.load(
    MODELS_DIR / "stage0_neural_network_scaler.joblib"
)

In [7]:
# Verify loaded models and preprocessing objects

loaded_objects = {
    "Logistic Regression": logistic_model,
    "Logistic Regression Scaler": logistic_scaler,
    "Random Forest": random_forest_model,
    "XGBoost": xgboost_model,
    "Neural Network": neural_network_model,
    "Neural Network Scaler": neural_network_scaler,
}

for name, obj in loaded_objects.items():
    print(f"{name}: {type(obj).__name__}")

print("\nModel verification")
print("-" * 40)

print(
    "Logistic Regression features:",
    logistic_model.n_features_in_,
)
print(
    "Random Forest features:",
    random_forest_model.n_features_in_,
)
print(
    "XGBoost features:",
    xgboost_model.n_features_in_,
)
print(
    "Neural Network input shape:",
    neural_network_model.input_shape,
)

print("\nScaler verification")
print("-" * 40)

print(
    "Logistic Regression scaler features:",
    logistic_scaler.n_features_in_,
)
print(
    "Neural Network scaler features:",
    neural_network_scaler.n_features_in_,
)

Logistic Regression: LogisticRegression
Logistic Regression Scaler: StandardScaler
Random Forest: RandomForestClassifier
XGBoost: XGBClassifier
Neural Network: Sequential
Neural Network Scaler: StandardScaler

Model verification
----------------------------------------
Logistic Regression features: 27
Random Forest features: 27
XGBoost features: 27
Neural Network input shape: (None, 27)

Scaler verification
----------------------------------------
Logistic Regression scaler features: 27
Neural Network scaler features: 27


## 5. Prepare Test Data

All models are evaluated on the same chronological test set. Logistic Regression and the neural network require standardized features, so we use their respective scalers fitted on the training data. Random Forest and XGBoost use the original feature values.

No scaler is fitted or refitted on the test set.

### 5.1 Standardize features for Logistic Regression and the Neural Network

In [8]:
X_test_logistic = logistic_scaler.transform(X_test)

X_test_neural_network = neural_network_scaler.transform(
    X_test
)

print("Logistic Regression input:", X_test_logistic.shape)
print("Random Forest input:", X_test.shape)
print("XGBoost input:", X_test.shape)
print("Neural Network input:", X_test_neural_network.shape)

Logistic Regression input: (919, 27)
Random Forest input: (919, 27)
XGBoost input: (919, 27)
Neural Network input: (919, 27)


Verify the input data:

In [9]:
assert X_test.shape == (919, 27)
assert X_test_logistic.shape == (919, 27)
assert X_test_neural_network.shape == (919, 27)

assert not X_test.isna().any().any()
assert np.isfinite(X_test.to_numpy()).all()

print("Test data verification passed.")

Test data verification passed.


## 6. Generate Test Predictions

We obtain positive-class probabilities from each saved model. Classification thresholds are fixed using the validation experiments conducted in the individual notebooks.

The thresholds affect precision, recall, F1, and accuracy, but not ROC-AUC or PR-AUC, which are calculated from predicted probabilities.

### 6.1 Obtain predicted probabilities

In [10]:
# Logistic Regression
logistic_test_probabilities = logistic_model.predict_proba(
    X_test_logistic
)[:, 1]

# Random Forest
random_forest_test_probabilities = random_forest_model.predict_proba(
    X_test
)[:, 1]

# XGBoost
xgboost_test_probabilities = xgboost_model.predict_proba(
    X_test
)[:, 1]

# Neural Network
neural_network_test_probabilities = neural_network_model.predict(
    X_test_neural_network,
    verbose=0,
).ravel()

### 6.2 Set the frozen classification thresholds

In [12]:
classification_thresholds = {
    "Logistic Regression": 0.50,
    "Random Forest": 0.20,
    "XGBoost": 0.20,
    "Neural Network": 0.30,
}

classification_thresholds

{'Logistic Regression': 0.5,
 'Random Forest': 0.2,
 'XGBoost': 0.2,
 'Neural Network': 0.3}

### 6.3 Convert probabilities into predictions

In [13]:
test_probabilities = {
    "Logistic Regression": logistic_test_probabilities,
    "Random Forest": random_forest_test_probabilities,
    "XGBoost": xgboost_test_probabilities,
    "Neural Network": neural_network_test_probabilities,
}

test_predictions = {
    name: (
        probabilities >= classification_thresholds[name]
    ).astype(int)
    for name, probabilities in test_probabilities.items()
}

for name, predictions in test_predictions.items():
    print(f"{name}: {np.bincount(predictions, minlength=2)}")

Logistic Regression: [873  46]
Random Forest: [417 502]
XGBoost: [403 516]
Neural Network: [539 380]


## 7. Test-Set Evaluation

We evaluate all four saved models using the same test observations and the classification thresholds selected during validation.

We report accuracy, balanced accuracy, precision, recall, F1, ROC-AUC, PR-AUC, and log loss. This allows us to compare both threshold-dependent classification performance and probability-ranking performance.

In [15]:
comparison_results = []

for model_name, probabilities in test_probabilities.items():

    predictions = test_predictions[model_name]

    metrics = classification_metrics(
        y_test,
        predictions,
        probabilities,
    )

    comparison_results.append({
        "model": model_name,
        "threshold": classification_thresholds[model_name],
        **metrics,
    })

comparison_df = pd.DataFrame(comparison_results)

comparison_df

,model,threshold,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,log_loss
0,Logistic Regression,0.5,0.799782,0.554704,0.565217,0.136842,0.220339,0.735268,0.409892,0.460211
1,Random Forest,0.2,0.588683,0.676532,0.312749,0.826316,0.453757,0.755693,0.421651,0.453763
2,XGBoost,0.2,0.571273,0.663613,0.302326,0.821053,0.441926,0.735947,0.390139,0.460589
3,Neural Network,0.3,0.660501,0.667320,0.339474,0.678947,0.452632,0.724894,0.381294,0.467498


### 7.2: Format the comparision table

In [16]:
metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "log_loss",
]

comparison_display = comparison_df.copy()

comparison_display[metric_columns] = (
    comparison_display[metric_columns].round(4)
)

comparison_display

,model,threshold,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,log_loss
0,Logistic Regression,0.5,0.7998,0.5547,0.5652,0.1368,0.2203,0.7353,0.4099,0.4602
1,Random Forest,0.2,0.5887,0.6765,0.3127,0.8263,0.4538,0.7557,0.4217,0.4538
2,XGBoost,0.2,0.5713,0.6636,0.3023,0.8211,0.4419,0.7359,0.3901,0.4606
3,Neural Network,0.3,0.6605,0.6673,0.3395,0.6789,0.4526,0.7249,0.3813,0.4675


### 7.3: Rank models by test F1

In [17]:
comparison_display.sort_values(
    "f1",
    ascending=False,
).reset_index(drop=True)

,model,threshold,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc,log_loss
0,Random Forest,0.2,0.5887,0.6765,0.3127,0.8263,0.4538,0.7557,0.4217,0.4538
1,Neural Network,0.3,0.6605,0.6673,0.3395,0.6789,0.4526,0.7249,0.3813,0.4675
2,XGBoost,0.2,0.5713,0.6636,0.3023,0.8211,0.4419,0.7359,0.3901,0.4606
3,Logistic Regression,0.5,0.7998,0.5547,0.5652,0.1368,0.2203,0.7353,0.4099,0.4602


## 8. Conclusions

In this notebook, we compared Logistic Regression, Random Forest, XGBoost, and a feedforward neural network on the same chronological test set covering 2023–2026.

### Summary of results

Random Forest achieved the highest test F1 score (0.4538), followed closely by the neural network (0.4526) and XGBoost (0.4419). Random Forest also achieved the highest ROC-AUC (0.7557) and PR-AUC (0.4217). Logistic Regression achieved high accuracy (0.7998), but its low recall (0.1368) and F1 score (0.2203) indicate that it identified relatively few positive cases.

The tree-based models and neural network were substantially more effective at identifying positive cases than Logistic Regression at their selected classification thresholds.

### Connection with the internship project

During the internship, the neural network was substantially less competitive than the leading tree-based model. In this public-SPY experiment, the neural network achieved an F1 score of 0.4526, placing it close to Random Forest (0.4538) and ahead of XGBoost (0.4419).

This contrast is encouraging: the neural network is competitive in the present experiment, unlike in the internship modelling. However, the two experiments differ in their data sources, feature representations, and target construction. We therefore cannot attribute the difference in neural-network performance to the addition of topological features.

A more direct investigation would have augmented the internship model's original anonymous features with TDA-derived features and compared performance under the same data split and evaluation protocol. The confidential nature of the internship modelling and the available time do not permit that experiment within this public repository.

### Interpretation and limitations

The present results establish a useful baseline for volatility prediction using public SPY data and a feature set combining conventional volatility statistics with TDA-derived descriptors. They do not establish the incremental predictive contribution of TDA, since all four models were trained using the same combined feature set.

The results also come from a single chronological test period. Although the neural network's performance is encouraging relative to the internship experience, further experiments would be needed to establish whether this behavior persists across other periods and model configurations.

### Next stage

Stage 0 provides a conventional machine-learning and feedforward neural-network baseline for the public-SPY task. The next stage will investigate recurrent neural networks (RNNs), which offer a natural way to model sequential financial observations.

The contribution of TDA features, and their interaction with different model architectures, remains an open research question for future work.